# **QUERY OPTIMIZER**

### IMPORT

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
import torch
import warnings


import sentencepiece as spm
import re
import json
from torch.utils.data import Dataset
import random

from torch.utils.tensorboard import SummaryWriter


In [ ]:
torch.set_float32_matmul_precision('high')
warnings.filterwarnings("ignore", category = FutureWarning)
writer = SummaryWriter(log_dir="runs/sql_training")


### DEVICE HANDLING

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device being used: {device}")


Device being used: cuda


#### HELPER

In [4]:
def safe_tensor(x):
    
    return x if torch.is_tensor(x) else torch.tensor(x, dtype = torch.float32).to(device)

def stack_tensor(x):
    
    return torch.stack(x).to(device)


### PARAMETER

In [ ]:
d_model = 256
d_ff = d_model * 4
num_heads = 4
dropout = 0.1
num_layers = 6
vocab_size = 375
tgt_size = vocab_size


### DECODER SINGLE LAYER


In [6]:
class decSingleLayer(nn.Module):
    
    def __init__(self, d_model, d_ff, num_heads, dropout):
        super(decSingleLayer, self).__init__()
        
        # start with layer norm for normalization
        
        self.layer_norm = nn.LayerNorm(d_model)
        
        # add masked multi head attention
        
        self.attention = nn.MultiheadAttention(d_model, num_heads, dropout, batch_first = True)
        

        # Feed forward network
        
        self.ffn_norm = nn.LayerNorm(d_model)
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.ffn_norm_2 = nn.LayerNorm(d_model)
        
        # dropout for regularization
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        
        # x -> Layer Norm
        
        x_2 = self.layer_norm(x)
        
        # Mask
        
        seq_length = x_2.size(1)
        
        mask = self.masked(x_2.device, seq_length)
        
        # Mask -> Multi
        
        attn, _ = self.attention(x_2, x_2, x_2, attn_mask = mask)
        
        # add residual network
        
        x = x + self.dropout(attn)
        
        
        # FFN
        
        x_2 = self.ffn_norm(x)
        x_2 = self.fc2(self.dropout(F.silu(self.fc1(x_2))))
        
        # add residual network
        
        x = x + self.dropout(x_2)
        
        return x
                
            
    def masked(self, device, seq_length):
        
        mask = torch.triu(torch.ones(seq_length, seq_length, device = device), diagonal = 1)
        
        mask = mask.masked_fill_(mask == 1, -torch.inf)
        
        return mask
        

#### INITIALIZE

In [7]:
Decoder_single_layer = decSingleLayer(d_model, d_ff, num_heads, dropout)
print(Decoder_single_layer)

print(" --------------------------------------- x --------------------------------------------------------")

parameter = sum(p.numel() for p in Decoder_single_layer.parameters())
print(f"Number of parameter model: {parameter}")


decSingleLayer(
  (layer_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
  )
  (ffn_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (fc1): Linear(in_features=256, out_features=1024, bias=True)
  (fc2): Linear(in_features=1024, out_features=256, bias=True)
  (ffn_norm_2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)
 --------------------------------------- x --------------------------------------------------------
Number of parameter model: 790272


### DECODER STACK

In [8]:
class decoderStack(nn.Module):
    
    def __init__(self, d_model, d_ff, num_heads, dropout, num_layers):
        super(decoderStack, self).__init__()
        
        # stack
        
        self.stack = nn.ModuleList([decSingleLayer(d_model, d_ff, num_heads, dropout) for _ in range(num_layers)])
        
    def forward(self, x):
        
        for layer in self.stack:
            
            x = layer(x)
            
        return x


#### INITIALIZE

In [9]:
DECODER = decoderStack(d_model, d_ff, num_heads, dropout, num_layers).to(device)
print(DECODER)

print(" ------------------------------------------------------- x -----------------------------------------------------")

parameter = sum([p.numel() for p in DECODER.parameters()])
print(f"Total parameter network: {parameter}")


decoderStack(
  (stack): ModuleList(
    (0-5): 6 x decSingleLayer(
      (layer_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attention): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (ffn_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (fc1): Linear(in_features=256, out_features=1024, bias=True)
      (fc2): Linear(in_features=1024, out_features=256, bias=True)
      (ffn_norm_2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
)
 ------------------------------------------------------- x -----------------------------------------------------
Total parameter network: 4741632


### POSITION ENCODING

In [10]:
def positional_encoding(seq_length, d_model):
    
    pe = torch.zeros(seq_length, d_model)
    position = torch.arange(0, seq_length, dtype = torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0 , d_model , 2).float() * (-math.log(10_000) / d_model))
    
    pe[: , 0::2] = torch.sin(position * div_term)
    pe[: , 1::2] = torch.cos(position * div_term)
    
    return pe
    

### MODEL STRUCTURE

In [11]:
class gpt_structure(nn.Module):
    
    def __init__(self, d_model, vocab_size, d_ff, num_heads, dropout, num_layers, tgt_size):
        super(gpt_structure, self).__init__()
        
        self.d_model = d_model
        
        # Embedding
        
        self.src_embed = nn.Embedding(vocab_size, d_model)
        
        # Decoder
        
        self.decoder = decoderStack(d_model, d_ff, num_heads, dropout, num_layers)
        
        # projection
        
        self.projection = nn.Linear(d_model, tgt_size)
        
        # regularization for projection
        
        self.apply(self.init_weight)
        
    def init_weight(self, m):
        
        if isinstance(m, nn.Linear):
            
            nn.init.kaiming_uniform_(m.weight)
            
            if m.bias is not None:
                
                nn.init.zeros_(m.bias)
                
        if isinstance(m, nn.MultiheadAttention):
            
            nn.init.xavier_uniform_(m.in_proj_weight)
            
            if m.in_proj_bias is not None:
                
                nn.init.zeros_(m.in_proj_bias)
                
            nn.init.xavier_uniform_(m.out_proj.weight)
            
            if m.out_proj.bias is not None:
                
                nn.init.zeros_(m.out_proj.bias)
                

                
    def forward(self, x):
        
        src_seq_length = x.size(1)
        
        # Embedding
        
        src = self.src_embed(x) * math.sqrt(self.d_model)
        
        # Positional encoding
        
        src_pe = positional_encoding(src_seq_length, self.d_model).to(src.device)
        
        # Decoder
        
        dec_out = self.decoder(src + src_pe)
        
        # projection
        
        out = self.projection(dec_out)
        
        return out


#### INITIALIZE

In [12]:
BODY = gpt_structure(d_model, vocab_size, d_ff, num_heads, dropout, num_layers, tgt_size).to(device)
print(BODY)

print("------------------------------------------------- x ------------------------------------------------")

parameter = sum([p.numel() for p in BODY.parameters()])
print(f"Total parameter model: {parameter}")


gpt_structure(
  (src_embed): Embedding(375, 256)
  (decoder): decoderStack(
    (stack): ModuleList(
      (0-5): 6 x decSingleLayer(
        (layer_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (ffn_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (fc1): Linear(in_features=256, out_features=1024, bias=True)
        (fc2): Linear(in_features=1024, out_features=256, bias=True)
        (ffn_norm_2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (projection): Linear(in_features=256, out_features=375, bias=True)
)
------------------------------------------------- x ------------------------------------------------
Total parameter model: 4934007


### LOSS FUNC & OPTIMIZER & SCHEDULER

In [13]:
# learning rate

lr = 1e-4

# Epochs

Epochs = 100
warmup_epoch = 40
pad_idx = 0

# loss func

loss_func = nn.CrossEntropyLoss(ignore_index = pad_idx)

# optimizer

optimizer = optim.AdamW(BODY.parameters(), lr)

# scheduler

warmup_Scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor = 0.3)
cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = Epochs - warmup_epoch - 10)

scheduler = optim.lr_scheduler.SequentialLR(optimizer, [warmup_Scheduler, cosine_scheduler], [warmup_epoch + 10])


### DATA PROCESSING

In [14]:
raw_text = open("synthetic_sql_dataset.txt").read()
entries = raw_text.split("-- ENTRY")

dataset = []

for entry in entries:
    prompt_match = re.search(r"Prompt \d+: (.*)", entry)
    query_match = re.search(r"Query: (.*)", entry)
    rule_match = re.search(r"Rule: (.*)", entry)
    explanation_match = re.search(r"Explanation: (.*)", entry)

    if not (prompt_match and query_match):
        continue  # skip broken entries

    prompt = prompt_match.group(1).strip()
    query = query_match.group(1).strip()
    rule = rule_match.group(1).strip() if rule_match else ""
    explanation = explanation_match.group(1).strip() if explanation_match else ""


    if len(query) < 10:
        continue
    if "??" in query:
        continue
    if "SUM(name)" in query: 
        continue

    sample = {
        "input": f"{prompt} Rule: {rule}",
        "output": query,
        "explanation": explanation
    }

    dataset.append(sample)

with open("clean_data.jsonl", "w") as f:
    for d in dataset:
        f.write(json.dumps(d) + "\n")

print(f"Saved {len(dataset)} clean samples")


Saved 495166 clean samples


### DATASET

In [15]:
def dataset(json_path):
    
    data_pairs = []
    
    with open(json_path, 'r', encoding='utf-8') as F:
        
        for line in F:
            
            entry = json.loads(line.strip())
            
            src_txt = entry['input']
            tgt_txt = entry['output'] + " " + entry['explanation']
            
            data_pairs.append((src_txt, tgt_txt))
    
    return data_pairs


In [16]:
data_pairs = dataset(r"C:\LLM & Agents\Query\clean_data.jsonl")


### TOKENIZER

In [17]:
def tokenized_function(data_pairs, sp_model):
    
    tokenized_pair = []
    
    for src_txt , tgt_txt in data_pairs:
        
        bos_id = sp_model.bos_id()
        eos_id = sp_model.eos_id()

        src_ids = [bos_id] + sp_model.encode(src_txt, out_type=int) + [eos_id]
        tgt_ids = [bos_id] + sp_model.encode(tgt_txt, out_type=int) + [eos_id]
        
        tokenized_pair.append((src_ids, tgt_ids))
        
    return tokenized_pair


In [18]:
spm.SentencePieceTrainer.train(
    input = "C:\LLM & Agents\Query\clean_data.jsonl",
    model_prefix = 'Query_Tokenizer',
    vocab_size = 375
)


sp = spm.SentencePieceProcessor()


In [19]:

sp.Load("C:\LLM & Agents\Query\Query_Tokenizer.model")

tokenized_pair = tokenized_function(data_pairs, sp)


### TORCH DATASET

In [20]:
class Query_PyDataset(Dataset):
    
    def __init__(self, tokenized_pairs, pad_idx, max_len):
        self.data = tokenized_pairs
        self.pad_idx = pad_idx
        self.max_len = max_len
        
    def pad(self, seq):
        seq = seq[:self.max_len]
        return seq + [self.pad_idx] * (self.max_len - len(seq))
    
    def __len__(self):
        
        return len(self.data)
    
    def __getitem__(self, idx):
        
        src_ids, tgt_ids = self.data[idx]
        
        full_seq = src_ids + tgt_ids
        
        full_seq = self.pad(full_seq)
        
        input_seq  = full_seq[:-1]
        target_seq = full_seq[1:]
        
        return (
            torch.tensor(input_seq, dtype=torch.long),
            torch.tensor(target_seq, dtype=torch.long)
        )


In [21]:
lengths = []

for src, tgt in tokenized_pair:
    
    lengths.append(len(src) + len(tgt))

print("Max:", max(lengths))
print("Avg:", sum(lengths)/len(lengths))

lengths_sorted = sorted(lengths)
print("90%:", lengths_sorted[int(0.9 * len(lengths))])
print("95%:", lengths_sorted[int(0.95 * len(lengths))])
print("99%:", lengths_sorted[int(0.99 * len(lengths))])


Max: 158
Avg: 104.1126026423462
90%: 124
95%: 130
99%: 139


### DATA LOADER

In [22]:
def split_data(data, train_ratio=0.8, val_ratio=0.1):
    
    random.shuffle(data)
    
    n = len(data)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    
    train_data = data[:train_end]
    val_data = data[train_end:val_end]
    test_data = data[val_end:]
    
    return train_data, val_data, test_data

train_pairs, val_pairs, test_pairs = split_data(tokenized_pair)


In [23]:
train_dataset = Query_PyDataset(train_pairs, pad_idx = 0, max_len = 140)
val_dataset   = Query_PyDataset(val_pairs, pad_idx = 0, max_len = 140)
test_dataset  = Query_PyDataset(test_pairs, pad_idx = 0, max_len = 140)


In [ ]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader  = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)


### TRAINING

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, loss_fn, epochs, device, scheduler):
    
    global_step = 0
    
    for epoch in range(epochs):
        
        # ===== TRAIN =====
        model.train()
        train_loss = 0
        
        for step, (input_seq, target_seq) in enumerate(train_loader):
            
            input_seq = input_seq.to(device)
            target_seq = target_seq.to(device)
            
            logits = model(input_seq)
            
            loss = loss_fn(
                logits.view(-1, logits.size(-1)),
                target_seq.view(-1)
            )
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()

            writer.add_scalar("Loss/Train_step", loss.item(), global_step)
            
            if step % 50 == 0:
                print(f"Epoch {epoch+1} | Step {step} | Loss: {loss.item():.4f}")
            
            global_step += 1
        
        avg_train_loss = train_loss / len(train_loader)
        
        # ===== VALIDATION =====
        model.eval()
        val_loss = 0
        
        with torch.no_grad():
            for input_seq, target_seq in val_loader:
                
                input_seq = input_seq.to(device)
                target_seq = target_seq.to(device)
                
                logits = model(input_seq)
                
                loss = loss_fn(
                    logits.view(-1, logits.size(-1)),
                    target_seq.view(-1)
                )
                
                val_loss += loss.item()
        
        avg_val_loss = val_loss / len(val_loader)
        

        writer.add_scalar("Loss/Train_epoch", avg_train_loss, epoch)
        writer.add_scalar("Loss/Val_epoch", avg_val_loss, epoch)
        
        print(f"\nEpoch {epoch+1} DONE | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}\n")
    
    writer.close()


In [29]:
for input_seq, target_seq in train_loader:
    
    print("Max input:", input_seq.max().item())
    print("Min input:", input_seq.min().item())
    
    print("Max target:", target_seq.max().item())
    print("Min target:", target_seq.min().item())
    
    print("Vocab size:", vocab_size)
    
    break


Max input: 271
Min input: 0
Max target: 271
Min target: 0
Vocab size: 375


In [30]:
train_model(BODY, train_loader, val_loader, optimizer, loss_func, Epochs, device, scheduler)


Epoch 1 | Step 0 | Loss: 67.6279
Epoch 1 | Step 50 | Loss: 37.7042
Epoch 1 | Step 100 | Loss: 20.8356
Epoch 1 | Step 150 | Loss: 11.8058
Epoch 1 | Step 200 | Loss: 6.1198
Epoch 1 | Step 250 | Loss: 3.0255
Epoch 1 | Step 300 | Loss: 1.9292
Epoch 1 | Step 350 | Loss: 1.5506
Epoch 1 | Step 400 | Loss: 1.3005
Epoch 1 | Step 450 | Loss: 1.1027
Epoch 1 | Step 500 | Loss: 0.9247
Epoch 1 | Step 550 | Loss: 0.8423
Epoch 1 | Step 600 | Loss: 0.7368
Epoch 1 | Step 650 | Loss: 0.6791
Epoch 1 | Step 700 | Loss: 0.6458
Epoch 1 | Step 750 | Loss: 0.6072
Epoch 1 | Step 800 | Loss: 0.5561
Epoch 1 | Step 850 | Loss: 0.5329
Epoch 1 | Step 900 | Loss: 0.4774
Epoch 1 | Step 950 | Loss: 0.4727
Epoch 1 | Step 1000 | Loss: 0.4733
Epoch 1 | Step 1050 | Loss: 0.4532
Epoch 1 | Step 1100 | Loss: 0.4290
Epoch 1 | Step 1150 | Loss: 0.4268
Epoch 1 | Step 1200 | Loss: 0.4217
Epoch 1 | Step 1250 | Loss: 0.4010
Epoch 1 | Step 1300 | Loss: 0.3854
Epoch 1 | Step 1350 | Loss: 0.3852
Epoch 1 | Step 1400 | Loss: 0.3723
Epoc

KeyboardInterrupt: 